# Google Search Ranking & Discoverability Capstone
## CTR / Engagement Opportunity Scoring

**Research question:** Which pages are most likely to under-capture clicks in the following month relative to their current CTR, given the search visibility and engagement signals available at the decision moment?

**Decision:** Which pages should an SEO/content reviewer prioritize for CTR or metadata investigation?

**Action:** Review the highest-ranked pages first; do not treat the score as proof that a page needs a specific change.

**Lane:** CTR / Engagement Opportunity Scoring.

This capstone uses a time-aware prediction setup: features are measured in month *t* and the outcome is measured in month *t+1*. March 2026 is training, April is validation, and May is the final test month whose outcome is June. Future outcome fields are never used as model features.


## 1. Data contract and public-safety rules

- **Warehouse:** FlyRank internship warehouse v20260703.
- **Source table:** `fact_content_daily_performance`.
- **Decision grain:** one pseudonymized client + content item + month after aggregating daily observations.
- **Development/test windows:** March–June 2026.
- **Outcome:** next-month CTR falls at least 20% from the current-month CTR, with at least 100 future impressions.
- **Features:** current impressions, current CTR, current average position, organic sessions, and engagement rate.
- **Excluded:** future-month clicks/CTR/impressions, label-derived trend fields, FlyRank composite/product flags, raw URLs, queries, domains, client names, and credentials.


In [ ]:
%pip install -q duckdb pandas numpy scikit-learn matplotlib

import os, json, warnings
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found. Add HF_TOKEN in Colab Secrets and enable notebook access.")

con = duckdb.connect()
con.execute(
    "CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)"

print("Warehouse connection ready.")


In [ ]:
# Aggregate only March-June 2026.
# June is used only as the held-out outcome for May.

monthly = con.sql(f'''
WITH agg AS (
    SELECT
        month,
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS impressions,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END) AS clicks,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_sum_position ELSE 0 END) AS sum_position,
        SUM(ga4_engaged_sessions) AS engaged_sessions,
        SUM(sessions_organic) AS sessions_organic,
        SUM(sessions_direct) AS sessions_direct,
        SUM(sessions_referral) AS sessions_referral,
        SUM(sessions_social) AS sessions_social,
        SUM(sessions_paid) AS sessions_paid,
        SUM(sessions_ai) AS sessions_ai,
        SUM(scroll_events) AS scroll_events
    FROM {FACT}
    WHERE month IN ('2026-03','2026-04','2026-05','2026-06')
    GROUP BY 1,2,3
)
SELECT
    *,
    CASE WHEN impressions > 0 THEN clicks * 1.0 / impressions END AS ctr,
    CASE WHEN impressions > 0 THEN sum_position * 1.0 / impressions END AS avg_position,
    CASE
        WHEN sessions_organic + sessions_direct + sessions_referral +
             sessions_social + sessions_paid + sessions_ai > 0
        THEN engaged_sessions * 1.0 /
             (sessions_organic + sessions_direct + sessions_referral +
              sessions_social + sessions_paid + sessions_ai)
    END AS engagement_rate
FROM agg
''').df()

print("Monthly decision rows:", len(monthly))
display(monthly.head())


## 2. Forward label

A page is an **opportunity** when its next-month CTR is at least 20% lower than its current CTR and the next month still contains at least 100 impressions.

This is a screening proxy for prioritization. It is not a causal claim.


In [ ]:
monthly["next_month"] = monthly["month"].map({
    "2026-03": "2026-04",
    "2026-04": "2026-05",
    "2026-05": "2026-06"
})

future = monthly[
    ["month","client_hash_id","content_hash_id","impressions","ctr"]
].rename(columns={
    "month":"future_month",
    "impressions":"future_impressions",
    "ctr":"future_ctr"
})

panel = monthly.merge(
    future,
    left_on=["next_month","client_hash_id","content_hash_id"],
    right_on=["future_month","client_hash_id","content_hash_id"],
    how="left"
)

panel["opportunity"] = (
    panel["future_impressions"].ge(100)
    & panel["ctr"].notna()
    & panel["future_ctr"].notna()
    & panel["future_ctr"].le(panel["ctr"] * 0.80)
)

print("Opportunity rate by decision month:")
display(
    panel.groupby("month")["opportunity"]
    .agg(["count","mean"])
    .rename(columns={"count":"rows","mean":"opportunity_rate"})
)


## 3. Leakage-safe features

The model only sees information known at the decision moment:

1. impressions
2. CTR
3. average position
4. organic sessions
5. engagement rate

It does **not** see the future-month outcome or FlyRank's precomputed product flags.


In [ ]:
FEATURES = [
    "impressions",
    "ctr",
    "avg_position",
    "sessions_organic",
    "engagement_rate"
]

model_df = panel.copy()

for col in FEATURES:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

model_df = model_df.dropna(
    subset=FEATURES + ["opportunity"]
).copy()

model_df = model_df[
    (model_df["impressions"] > 0) &
    (model_df["avg_position"] > 0) &
    np.isfinite(model_df["ctr"]) &
    np.isfinite(model_df["engagement_rate"])
].copy()

train = model_df[model_df["month"] == "2026-03"].copy()
valid = model_df[model_df["month"] == "2026-04"].copy()
test = model_df[model_df["month"] == "2026-05"].copy()

print("Train:", len(train))
print("Validation:", len(valid))
print("Test:", len(test))


## 4. Baseline

The baseline is deliberately transparent: within each position bucket, calculate how far current CTR is below the bucket median, then weight the gap by `log(1 + impressions)`.

The same held-out May→June test rows are used for the baseline and the learned model.


In [ ]:
def add_baseline_score(frame):
    out = frame.copy()

    out["position_bucket"] = pd.cut(
        out["avg_position"],
        bins=[0,3,5,10,20,50,np.inf],
        labels=["1-3","4-5","6-10","11-20","21-50","51+"],
        include_lowest=True
    )

    medians = (
        out.groupby("position_bucket", observed=False)["ctr"]
        .median()
        .rename("bucket_median_ctr")
    )

    out = out.join(medians, on="position_bucket")
    out["ctr_gap"] = (
        out["bucket_median_ctr"] - out["ctr"]
    ).clip(lower=0)

    out["baseline_score"] = (
        out["ctr_gap"] * np.log1p(out["impressions"])
    )

    return out

def precision_at_k(y, scores, k=50):
    y = np.asarray(y)
    scores = np.asarray(scores)
    k = min(k, len(y))
    idx = np.argsort(scores)[::-1][:k]
    return float(y[idx].mean())

test_scored = add_baseline_score(test)

baseline_p50 = precision_at_k(
    test_scored["opportunity"],
    test_scored["baseline_score"],
    50
)

print("Baseline Precision@50:", round(baseline_p50, 4))


## 5. Learned model

A Random Forest classifier combines the five current-period signals nonlinearly. It outputs an opportunity probability that becomes the ranking score.


In [ ]:
X_train = train[FEATURES]
y_train = train["opportunity"].astype(int)

X_valid = valid[FEATURES]
y_valid = valid["opportunity"].astype(int)

X_test = test[FEATURES]
y_test = test["opportunity"].astype(int)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

valid_prob = model.predict_proba(X_valid)[:,1]
test_prob = model.predict_proba(X_test)[:,1]

model_p50_valid = precision_at_k(y_valid, valid_prob, 50)
model_p50_test = precision_at_k(y_test, test_prob, 50)

print("Validation Precision@50:", round(model_p50_valid, 4))
print("Test Precision@50:", round(model_p50_test, 4))

if y_test.nunique() == 2:
    print("Test ROC-AUC:", round(roc_auc_score(y_test, test_prob), 4))
    print("Test PR-AUC:", round(average_precision_score(y_test, test_prob), 4))

lift = model_p50_test / baseline_p50 if baseline_p50 > 0 else np.nan
print("Precision@50 lift:", round(lift, 2), "x")


In [ ]:
# Results table

results = pd.DataFrame({
    "method": ["Baseline", "Random Forest"],
    "precision_at_50": [baseline_p50, model_p50_test]
})

display(results)

# Feature importance
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance)

os.makedirs("work/outputs/capstone", exist_ok=True)

plt.figure(figsize=(8,5))
plt.barh(importance["feature"][::-1], importance["importance"][::-1])
plt.xlabel("Feature importance")
plt.title("Signals used by the opportunity model")
plt.tight_layout()
plt.savefig("work/outputs/capstone/feature_importance.png", dpi=180)
plt.show()

# Precision@k chart
ks = [10,25,50,100]
baseline_curve = [
    precision_at_k(test_scored["opportunity"], test_scored["baseline_score"], k)
    for k in ks
]
model_curve = [
    precision_at_k(y_test, test_prob, k)
    for k in ks
]

plt.figure(figsize=(8,5))
plt.plot(ks, baseline_curve, marker="o", label="Baseline")
plt.plot(ks, model_curve, marker="o", label="Random Forest")
plt.xlabel("Top-k pages")
plt.ylabel("Precision")
plt.title("Held-out test performance")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("work/outputs/capstone/precision_at_k.png", dpi=180)
plt.show()


## 6. Ranked recommendations

The recommendation queue is the held-out May decision set ranked by model probability. The action is **review**, not automatic editing. Human review should check search intent, SERP context, seasonality, and page quality before acting.


In [ ]:
recommendations = test.copy()
recommendations["model_score"] = test_prob

threshold = recommendations["model_score"].quantile(0.90)

recommendations["action"] = np.where(
    recommendations["model_score"] >= threshold,
    "PRIORITIZE_CTR_REVIEW",
    "MONITOR"
)

recommendations["reason_code"] = np.select(
    [
        (recommendations["ctr"] < 0.02) &
        (recommendations["avg_position"] <= 20),
        (recommendations["impressions"] >= recommendations["impressions"].median()) &
        (recommendations["avg_position"] <= 20)
    ],
    [
        "LOW_CTR_VISIBLE_PAGE",
        "HIGH_VISIBILITY_REVIEW"
    ],
    default="MODEL_PRIORITY"
)

recommendations = recommendations.sort_values(
    "model_score", ascending=False
).reset_index(drop=True)

recommendations["rank"] = np.arange(1, len(recommendations)+1)

# Public-safe queue: no client/content hashes.
public_queue = recommendations[
    [
        "rank","model_score","action","reason_code",
        "impressions","ctr","avg_position"
    ]
]

public_queue.to_csv(
    "work/outputs/capstone/ranked_recommendations.csv",
    index=False
)

display(public_queue.head(20))


In [ ]:
metrics = {
    "lane": "CTR / Engagement Opportunity Scoring",
    "question": "Which pages are most likely to under-capture clicks in the following month?",
    "train": "2026-03 -> 2026-04",
    "validation": "2026-04 -> 2026-05",
    "test": "2026-05 -> 2026-06",
    "baseline_precision_at_50": float(baseline_p50),
    "model_precision_at_50": float(model_p50_test),
    "precision_at_50_lift": float(lift),
    "train_rows": int(len(train)),
    "validation_rows": int(len(valid)),
    "test_rows": int(len(test)),
    "features": FEATURES
}

with open("work/outputs/capstone/metrics.json","w") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))
print()
print("CAPSTONE ANALYSIS COMPLETE")


## 7. Honest framing

**Observed:** On the held-out May→June test window, the model and the transparent baseline can be compared at the same top-k operating points.

**Directional:** A higher model score means the model estimates a higher probability of the defined next-month opportunity proxy. It does not explain why the page changes.

**Decision-support:** The ranking is intended to prioritize human review.

**Not claimed:** This work does not prove causality, prove Google's ranking algorithm, or guarantee that editing a recommended page will improve CTR.


## 8. Generate the public research paper

Run the next cell **after all model/result cells succeed**. It creates `docs/index.html`, copies the two charts into `docs/assets/`, and writes the required `submission/paper_url.txt`.

Do not put row-level client/content identifiers into the public paper.


In [ ]:
# ============================================================
# 8. BUILD THE DEPLOYED RESEARCH PAPER
# ============================================================

import os, shutil, html, json

os.makedirs("docs/assets", exist_ok=True)

for filename in ["feature_importance.png", "precision_at_k.png"]:
    src = f"work/outputs/capstone/{filename}"
    if os.path.exists(src):
        shutil.copy2(src, f"docs/assets/{filename}")

baseline_text = f"{baseline_p50:.3f}"
model_text = f"{model_p50_test:.3f}"
lift_text = f"{lift:.2f}x"
train_text = f"{len(train):,}"
valid_text = f"{len(valid):,}"
test_text = f"{len(test):,}"

html_page = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Google Search Ranking & Discoverability — CTR Opportunity Scoring</title>
<style>
:root {{ --ink:#17202a; --muted:#64748b; --paper:#f7f7f3; --card:#ffffff; --line:#dfe4ea; --accent:#0f766e; }}
* {{ box-sizing:border-box; }}
body {{ margin:0; background:var(--paper); color:var(--ink); font:16px/1.65 system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif; }}
main {{ max-width:980px; margin:auto; padding:56px 24px 90px; }}
.eyebrow {{ color:var(--accent); font-weight:800; letter-spacing:.08em; text-transform:uppercase; font-size:.78rem; }}
h1 {{ font-size:clamp(2.2rem,5vw,4.5rem); line-height:1.02; max-width:850px; margin:.5rem 0 1rem; }}
h2 {{ margin-top:3rem; font-size:1.65rem; }}
.lede {{ font-size:1.18rem; color:#334155; max-width:800px; }}
.card {{ background:var(--card); border:1px solid var(--line); border-radius:18px; padding:24px; margin:18px 0; }}
.grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(180px,1fr)); gap:14px; }}
.metric {{ background:#fff; border:1px solid var(--line); border-radius:16px; padding:20px; }}
.metric strong {{ display:block; font-size:2rem; line-height:1; margin-bottom:8px; }}
.metric span {{ color:var(--muted); font-size:.9rem; }}
img {{ max-width:100%; border:1px solid var(--line); border-radius:14px; background:#fff; }}
code {{ background:#eef2f2; padding:.15em .35em; border-radius:5px; }}
.small {{ color:var(--muted); font-size:.92rem; }}
a {{ color:var(--accent); }}
footer {{ margin-top:60px; padding-top:20px; border-top:1px solid var(--line); color:var(--muted); }}
</style>
</head>
<body>
<main>
<div class="eyebrow">Machine Learning Capstone · FlyRank Internship</div>
<h1>Which pages are most likely to under-capture clicks next month?</h1>
<p class="lede">A leakage-aware, time-split ranking model for CTR / Engagement Opportunity Scoring on the FlyRank search warehouse.</p>

<div class="card">
<h2 style="margin-top:0">Abstract</h2>
<p><b>Question.</b> Can current search visibility and engagement signals identify pages that are likely to experience a meaningful CTR decline in the following month?</p>
<p><b>Method.</b> I aggregated the daily warehouse to a client-content-month decision table, defined a next-month opportunity proxy, compared a transparent CTR-vs-position baseline with a Random Forest, and evaluated both on the same held-out May→June window.</p>
<p><b>Result.</b> On the held-out test set, the baseline reached Precision@50 of <b>{baseline_text}</b> and the model reached <b>{model_text}</b>, a <b>{lift_text}</b> precision lift.</p>
<p><b>Interpretation.</b> The model is useful as a review-prioritization signal, not as a causal explanation of ranking or CTR changes.</p>
<p><b>Action.</b> Review the highest-ranked pages first, then use human judgment to inspect search intent, SERP context, seasonality, and page quality.</p>
</div>

<h2>Introduction / Problem statement</h2>
<p>SEO teams cannot manually investigate every visible page every month. The decision is therefore a prioritization problem: given information available now, which pages deserve review first because their click capture may weaken in the next measurement window?</p>
<p>This project focuses on CTR opportunity scoring. The output is a ranked review queue rather than an automatic recommendation to rewrite a page.</p>

<h2>Data</h2>
<div class="card">
<ul>
<li><b>Release:</b> FlyRank internship warehouse v20260703.</li>
<li><b>Source:</b> <code>fact_content_daily_performance</code>.</li>
<li><b>Grain:</b> daily client + content observations aggregated to monthly decision rows.</li>
<li><b>Windows:</b> March 2026 training, April validation, May test; June supplies the held-out May outcome.</li>
<li><b>Excluded:</b> future outcomes, trend/product flags, raw URLs, domains, queries, client names, and credentials.</li>
</ul>
</div>

<h2>Methodology</h2>
<p>The opportunity label is a practical proxy: next-month CTR is at least 20% lower than current-month CTR and the next month contains at least 100 impressions. The model uses five current-period features: impressions, CTR, average position, organic sessions, and engagement rate.</p>
<p>The baseline ranks by current CTR weakness relative to the median CTR of the page's position bucket, weighted by log impressions. The learned model is a Random Forest with 300 trees, depth capped at 8, minimum leaf size 20, and balanced class weighting.</p>
<p>The split is strictly time-aware: March→April for training, April→May for validation, and May→June for the final test. No future-month outcome fields enter the features.</p>

<h2>Results</h2>
<div class="grid">
<div class="metric"><strong>{baseline_text}</strong><span>Baseline Precision@50</span></div>
<div class="metric"><strong>{model_text}</strong><span>Model Precision@50</span></div>
<div class="metric"><strong>{lift_text}</strong><span>Precision@50 lift</span></div>
<div class="metric"><strong>{test_text}</strong><span>Held-out test rows</span></div>
</div>

<div class="card">
<h3>Held-out ranking performance</h3>
<img src="assets/precision_at_k.png" alt="Precision at k comparison">
</div>

<div class="card">
<h3>Signals used by the model</h3>
<img src="assets/feature_importance.png" alt="Random Forest feature importance">
</div>

<h2>Limitations & honest framing</h2>
<p><b>Observed:</b> the held-out test comparison measures how concentrated the defined opportunity cases are near the top of the ranked list.</p>
<p><b>Directional:</b> a high score means the model estimates a higher probability of the defined proxy, not that a page is definitely underperforming.</p>
<p><b>Decision-support:</b> the queue is intended to prioritize human investigation.</p>
<p>This analysis does not establish causality, does not prove Google's ranking algorithm, and does not guarantee that changing a recommended page will improve CTR. Search intent, SERP composition, seasonality, and measurement availability can all affect the observed outcome.</p>

<h2>Ranked recommendations</h2>
<p>The model's top-ranked pages form a review queue. The operational playbook is:</p>
<ol>
<li>Start with the highest model scores.</li>
<li>Confirm that the page has meaningful search visibility.</li>
<li>Inspect search intent and SERP context.</li>
<li>Check whether title/snippet or content alignment plausibly explains weak click capture.</li>
<li>Only then decide whether to test metadata, content changes, or monitoring.</li>
</ol>

<h2>Reproducibility</h2>
<p>The complete analysis notebook is in the public repository:</p>
<p><a href="https://github.com/ujjwalkpandey/flyrank-internship">github.com/ujjwalkpandey/flyrank-internship</a></p>
<p>Capstone notebook: <code>work/notebooks/capstone_ctr_opportunity.ipynb</code>.</p>
<p class="small">Training rows: {train_text} · Validation rows: {valid_text} · Test rows: {test_text}</p>

<h2>Acknowledgments & data credit</h2>
<p>Built on the FlyRank ML Internship dataset. Data credit: <a href="https://flyrank.ai">FlyRank</a>.</p>

<footer>
Google Search Ranking & Discoverability Capstone · CTR / Engagement Opportunity Scoring<br>
Public-safe aggregate presentation. No client names, domains, URLs, queries, or private data.
</footer>
</main>
</body>
</html>
"""

with open("docs/index.html", "w", encoding="utf-8") as f:
    f.write(html_page)

with open("docs/.nojekyll", "w") as f:
    f.write("")

os.makedirs("submission", exist_ok=True)
with open("submission/paper_url.txt", "w") as f:
    f.write("https://ujjwalkpandey.github.io/flyrank-internship/\n")

print("DEPLOYMENT FILES CREATED")
print("docs/index.html")
print("docs/assets/feature_importance.png")
print("docs/assets/precision_at_k.png")
print("submission/paper_url.txt")
print("Paper URL: https://ujjwalkpandey.github.io/flyrank-internship/")
